In [1]:
# 1. Install huggingface hub
# !pip install -U huggingface_hub

In [2]:
!pip install datasets pyarrow pillow

In [ ]:
from huggingface_hub import login

# Pass your token directly as a string
login(token="")

In [4]:
import os
import shutil
from datasets import load_dataset
from huggingface_hub import hf_hub_download

# --- CONFIGURATION ---
# Set this to 'nondisjoint' or 'disjoint' depending on which split you want to test/train on
SPLIT_TYPE = "nondisjoint"

BASE_DIR = "data/polyvore_outfits"
IMG_DIR = os.path.join(BASE_DIR, "images")
TARGET_METADATA_DIR = os.path.join(BASE_DIR, SPLIT_TYPE)

os.makedirs(IMG_DIR, exist_ok=True)
os.makedirs(TARGET_METADATA_DIR, exist_ok=True)

REPO_ID = "mvasil/polyvore-outfits"

# 1. Download Metadata and Task Files from the HF Subfolder
print(f"Downloading evaluation and metadata files for: {SPLIT_TYPE}...")
files_to_download = [
    "train.json", "valid.json", "test.json", "typespaces.p",
    "compatibility_train.txt", "compatibility_valid.txt", "compatibility_test.txt",
    "fill_in_blank_train.json", "fill_in_blank_valid.json", "fill_in_blank_test.json"
]

for file_name in files_to_download:
    try:
        downloaded_path = hf_hub_download(
            repo_id=REPO_ID,
            filename=f"{SPLIT_TYPE}/{file_name}",
            repo_type="dataset"
        )
        # Move it to the local directory where the PyTorch script expects it
        shutil.copy(downloaded_path, os.path.join(TARGET_METADATA_DIR, file_name))
        print(f" Successfully set up {file_name}")
    except Exception as e:
        print(f" Could not download {file_name}: {e}")

# 2. Extract Images from Parquet Splits
print("\nExtracting images from Parquet splits... This may take a while.")
splits = ["train", "validation", "test"]

for split in splits:
    print(f"Processing {split} split...")
    # Load dataset via streaming or direct download to extract images
    dataset = load_dataset(REPO_ID, SPLIT_TYPE, split=split)

    for row in dataset:
        # Expected Parquet schema columns include item info and image byte/PIL objects
        # We save images using their ID naming convention: item_id.jpg
        if 'items' in row and 'images' in row:
            for item, img in zip(row['items'], row['images']):
                item_id = item.get('item_id')
                if item_id:
                    img_path = os.path.join(IMG_DIR, f"{item_id}.jpg")
                    if not os.path.exists(img_path):
                        # Convert PIL image representation to RGB and save as JPG
                        img.convert("RGB").save(img_path)

print("\nDataset fully reconstructed for legacy code execution!")

nondisjoint/train.json:   0%|          | 0.00/31.8M [00:00<?, ?B/s]

 Successfully set up train.json


valid.json: 0.00B [00:00, ?B/s]

 Successfully set up valid.json


test.json: 0.00B [00:00, ?B/s]

...

data/nondisjoint/test.parquet:   0%|          | 0.00/447M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/204679 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/25132 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/47854 [00:00<?, ? examples/s]

Processing validation split...
Processing test split...

Dataset fully reconstructed for legacy code execution!


In [5]:
import os
import io
from PIL import Image
from datasets import load_dataset

IMG_DIR = "data/polyvore_outfits/images"
os.makedirs(IMG_DIR, exist_ok=True)
REPO_ID = "mvasil/polyvore-outfits"
SPLIT_TYPE = "nondisjoint"

print("Extracting images from Parquet splits...")
splits = ["train", "validation", "test"]

total_extracted = 0

for split in splits:
    print(f"\nLoading {split} split...")
    dataset = load_dataset(REPO_ID, SPLIT_TYPE, split=split)
    cols = dataset.column_names

    for row in dataset:
        if 'image' in cols and 'item_id' in cols:
            img_data = row['image']
            item_id = row['item_id']

            if img_data and item_id:
                img_path = os.path.join(IMG_DIR, f"{item_id}.jpg")

                if not os.path.exists(img_path):
                    # --- THE FIX: Handle raw bytes dictionary ---
                    if isinstance(img_data, dict) and 'bytes' in img_data:
                        # Convert raw bytes into a PIL Image
                        img = Image.open(io.BytesIO(img_data['bytes']))
                    else:
                        # Fallback if it's already a PIL Image
                        img = img_data

                    # Save the image
                    img.convert("RGB").save(img_path)
                    total_extracted += 1

print(f"\n✅ Done! Successfully extracted {total_extracted} images to {IMG_DIR}.")

Extracting images from Parquet splits...

Loading train split...

Loading validation split...

Loading test split...

✅ Done! Successfully extracted 251008 images to data/polyvore_outfits/images.


In [6]:
import os
import shutil
from huggingface_hub import hf_hub_download

# Create the directory just in case it doesn't exist
os.makedirs("data/polyvore_outfits", exist_ok=True)

print("Downloading master metadata file...")

# This uses your existing Hugging Face login token!
download_path = hf_hub_download(
    repo_id="mvasil/polyvore-outfits",
    filename="polyvore_item_metadata.json",
    repo_type="dataset"
)

# Move it to exactly where the main.py script expects it
shutil.copy(download_path, "data/polyvore_outfits/polyvore_item_metadata.json")

print("Metadata downloaded successfully! You are ready to train.")

polyvore_item_metadata.json:   0%|          | 0.00/105M [00:00<?, ?B/s]

Metadata downloaded successfully! You are ready to train.


In [7]:
%%writefile Resnet_18.py
import torch
import torch.nn as nn
import math
from torch.hub import load_state_dict_from_url # Modern equivalent of model_zoo

__all__ = ['ResNet', 'resnet18']

model_urls = {
    'resnet18': 'https://download.pytorch.org/models/resnet18-5c106cde.pth',
}

def conv3x3(in_planes, out_planes, stride=1):
    """3x3 convolution with padding"""
    return nn.Conv2d(in_planes, out_planes, kernel_size=3, stride=stride,
                     padding=1, bias=False)

class BasicBlock(nn.Module):
    expansion = 1

    def __init__(self, inplanes, planes, stride=1, downsample=None):
        super(BasicBlock, self).__init__()
        self.conv1 = conv3x3(inplanes, planes, stride)
        self.bn1 = nn.BatchNorm2d(planes)
        self.relu = nn.ReLU(inplace=True)
        self.conv2 = conv3x3(planes, planes)
        self.bn2 = nn.BatchNorm2d(planes)
        self.downsample = downsample
        self.stride = stride

    def forward(self, x):
        residual = x

        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)

        out = self.conv2(out)
        out = self.bn2(out)

        if self.downsample is not None:
            residual = self.downsample(x)

        out += residual
        out = self.relu(out)

        return out

class ResNet(nn.Module):

    def __init__(self, block, layers, embedding_size=64):
        self.inplanes = 64
        super(ResNet, self).__init__()
        self.conv1 = nn.Conv2d(3, 64, kernel_size=7, stride=2, padding=3,
                               bias=False)
        self.bn1 = nn.BatchNorm2d(64)
        self.relu = nn.ReLU(inplace=True)
        self.maxpool = nn.MaxPool2d(kernel_size=3, stride=2, padding=1)
        self.layer1 = self._make_layer(block, 64, layers[0])
        self.layer2 = self._make_layer(block, 128, layers[1], stride=2)
        self.layer3 = self._make_layer(block, 256, layers[2], stride=2)
        self.avgpool = nn.AvgPool2d(7)
        self.fc_embed = nn.Linear(256 * block.expansion, embedding_size)

        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                n = m.kernel_size[0] * m.kernel_size[1] * m.out_channels
                m.weight.data.normal_(0, math.sqrt(2. / n))
            elif isinstance(m, nn.BatchNorm2d):
                m.weight.data.fill_(1)
                m.bias.data.zero_()

    def _make_layer(self, block, planes, blocks, stride=1):
        downsample = None
        if stride != 1 or self.inplanes != planes * block.expansion:
            downsample = nn.Sequential(
                nn.Conv2d(self.inplanes, planes * block.expansion,
                          kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(planes * block.expansion),
            )

        layers = []
        layers.append(block(self.inplanes, planes, stride, downsample))
        self.inplanes = planes * block.expansion
        for i in range(1, blocks):
            layers.append(block(self.inplanes, planes))

        return nn.Sequential(*layers)

    def forward(self, x):
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu(x)
        x = self.maxpool(x)

        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)

        x = self.avgpool(x)
        x = torch.flatten(x, 1) # Modern replacement for x.view(x.size(0), -1)
        x = self.fc_embed(x)

        return x

def resnet18(pretrained=False, **kwargs):
    """Constructs a ResNet-18 model.
    Args:
        pretrained (bool): If True, returns a model pre-trained on ImageNet
    """
    model = ResNet(BasicBlock, [2, 2, 2], **kwargs)
    if pretrained:
        state = model.state_dict()
        loaded_state_dict = load_state_dict_from_url(model_urls['resnet18'])
        for k in loaded_state_dict:
            if k in state:
                state[k] = loaded_state_dict[k]
        model.load_state_dict(state)
    return model

Writing Resnet_18.py


polyvore dataset

In [8]:
%%writefile polyvore_outfits.py
from PIL import Image
import os
import os.path
import torch.utils.data
import torchvision.transforms as transforms
import numpy as np
import json
import torch
import pickle
import itertools
from sklearn.metrics import roc_auc_score

def default_image_loader(path):
    return Image.open(path).convert('RGB')

def parse_iminfo(question, im2index, id2im, gt=None):
    questions = []
    is_correct = np.zeros(len(question), dtype=bool)

    for index, im_id in enumerate(question):
        set_id = im_id.split('_')[0]
        if gt is None:
            gt = set_id

        im = id2im[im_id]
        questions.append((im2index[im], im))
        is_correct[index] = (set_id == gt)

    return questions, is_correct, gt

def load_typespaces(rootdir, rand_typespaces, num_rand_embed):
    typespace_fn = os.path.join(rootdir, 'typespaces.p')

    if os.path.isfile(typespace_fn):
        typespaces = pickle.load(open(typespace_fn, 'rb'))
    else:
        print(f"Warning: {typespace_fn} missing. Auto-generating typespaces...")
        meta_fn = os.path.join(os.path.dirname(rootdir), 'polyvore_item_metadata.json')
        meta_data = json.load(open(meta_fn, 'r'))

        categories = set(v['semantic_category'] for v in meta_data.values())
        typespaces = list(itertools.combinations(categories, 2)) + [(c, c) for c in categories]

        os.makedirs(rootdir, exist_ok=True)
        pickle.dump(typespaces, open(typespace_fn, 'wb'))

    if not rand_typespaces:
        ts = {}
        for index, t in enumerate(typespaces):
            ts[t] = index
        return ts

    fn = os.path.join(rootdir, 'typespaces_rand_%i.p' % num_rand_embed)
    if os.path.isfile(fn):
        typespaces = pickle.load(open(fn, 'rb'))
    else:
        spaces = np.random.permutation(len(typespaces))
        width = np.ceil(len(spaces) / float(num_rand_embed))
        ts = {}
        for index, t in enumerate(spaces):
            ts[typespaces[t]] = int(np.floor(index / width))
        pickle.dump(ts, open(fn, 'wb'))

    return ts

def load_compatibility_questions(fn, im2index, id2im):
    with open(fn, 'r') as f:
        lines = f.readlines()

    compatibility_questions = []
    for line in lines:
        data = line.strip().split()
        compat_question, _, _ = parse_iminfo(data[1:], im2index, id2im)
        compatibility_questions.append((compat_question, int(data[0])))

    return compatibility_questions

def load_fitb_questions(fn, im2index, id2im):
    data = json.load(open(fn, 'r'))
    questions = []
    for item in data:
        question = item['question']
        q_index, _, gt = parse_iminfo(question, im2index, id2im)
        answer = item['answers']
        a_index, is_correct, _ = parse_iminfo(answer, im2index, id2im, gt)
        questions.append((q_index, a_index, is_correct))

    return questions

class TripletImageLoader(torch.utils.data.Dataset):
    def __init__(self, args, split, meta_data, text_dim=None, transform=None, loader=default_image_loader):
        rootdir = os.path.join(args.datadir, 'polyvore_outfits', args.polyvore_split)
        self.impath = os.path.join(args.datadir, 'polyvore_outfits', 'images')
        self.is_train = split == 'train'
        data_json = os.path.join(rootdir, '%s.json' % split)

        with open(data_json, 'r') as f:
            outfit_data = json.load(f)

        im2type = {}
        category2ims = {}
        imnames = set()
        id2im = {}
        for outfit in outfit_data:
            outfit_id = outfit['set_id']
            for item in outfit['items']:
                im = item['item_id']
                category = meta_data[im]['semantic_category']
                im2type[im] = category

                if category not in category2ims:
                    category2ims[category] = {}

                if outfit_id not in category2ims[category]:
                    category2ims[category][outfit_id] = []

                category2ims[category][outfit_id].append(im)
                id2im['%s_%i' % (outfit_id, item['index'])] = im
                imnames.add(im)

        imnames = list(imnames)
        im2index = {}
        for index, im in enumerate(imnames):
            im2index[im] = index

        self.data = outfit_data
        self.imnames = imnames
        self.im2type = im2type
        self.typespaces = load_typespaces(rootdir, args.rand_typespaces, args.num_rand_embed)
        self.transform = transform
        self.loader = loader
        self.split = split

        if self.is_train:
            self.text_feat_dim = text_dim
            self.desc2vecs = {}
            featfile = os.path.join(rootdir, 'train_hglmm_pca6000.txt')

            if os.path.exists(featfile):
                with open(featfile, 'r') as f:
                    for line in f:
                        line = line.strip()
                        if not line:
                            continue
                        vec = line.split(',')
                        label = ','.join(vec[:-self.text_feat_dim])
                        vec = np.array([float(x) for x in vec[-self.text_feat_dim:]], dtype=np.float32)
                        self.desc2vecs[label] = vec
            else:
                pass # Proceeding with Vision-Only features as warned

            self.im2desc = {}
            for im in imnames:
                desc = meta_data[im]['title']
                if not desc:
                    desc = meta_data[im]['url_name']

                desc = desc.replace('\n', '').encode('ascii', 'ignore').decode('ascii').strip().lower()

                if desc and desc in self.desc2vecs:
                    self.im2desc[im] = desc

            pos_pairs = []
            max_items = 0
            for outfit in outfit_data:
                items = outfit['items']
                cnt = len(items)
                max_items = max(cnt, max_items)
                outfit_id = outfit['set_id']
                for j in range(cnt - 1):
                    for k in range(j + 1, cnt):
                        pos_pairs.append([outfit_id, items[j]['item_id'], items[k]['item_id']])

            self.pos_pairs = pos_pairs
            self.category2ims = category2ims
            self.max_items = max_items
        else:
            fn = os.path.join(rootdir, 'fill_in_blank_%s.json' % split)
            self.fitb_questions = load_fitb_questions(fn, im2index, id2im)
            fn = os.path.join(rootdir, 'compatibility_%s.txt' % split)
            self.compatibility_questions = load_compatibility_questions(fn, im2index, id2im)

    def load_train_item(self, image_id):
        imfn = os.path.join(self.impath, '%s.jpg' % image_id)
        img = self.loader(imfn)
        if self.transform is not None:
            img = self.transform(img)

        if image_id in self.im2desc:
            text = self.im2desc[image_id]
            text_features = self.desc2vecs[text]
            has_text = 1.0
        else:
            text_features = np.zeros(self.text_feat_dim, dtype=np.float32)
            has_text = 0.0

        has_text = np.float32(has_text)
        item_type = self.im2type[image_id]
        return img, text_features, has_text, item_type

    def sample_negative(self, outfit_id, item_id, item_type):
        item_out = item_id
        candidate_sets = list(self.category2ims[item_type].keys())
        attempts = 0
        while item_out == item_id and attempts < 100:
            choice = np.random.choice(candidate_sets)
            items = self.category2ims[item_type][choice]
            item_index = np.random.choice(range(len(items)))
            item_out = items[item_index]
            attempts += 1

        return item_out

    def get_typespace(self, anchor, pair):
        query = (anchor, pair)
        if query not in self.typespaces:
            query = (pair, anchor)
        return self.typespaces[query]

    def test_compatibility(self, embeds, metric):
        scores = []
        labels = np.zeros(len(self.compatibility_questions), dtype=np.int32)

        for index, (outfit, label) in enumerate(self.compatibility_questions):
            labels[index] = label
            n_items = len(outfit)
            outfit_score = 0.0
            num_comparisons = 0.0

            for i in range(n_items - 1):
                item1, img1 = outfit[i]
                type1 = self.im2type[img1]
                for j in range(i + 1, n_items):
                    item2, img2 = outfit[j]
                    type2 = self.im2type[img2]
                    condition = self.get_typespace(type1, type2)
                    embed1 = embeds[item1][condition].unsqueeze(0)
                    embed2 = embeds[item2][condition].unsqueeze(0)

                    if metric is None:
                        outfit_score += torch.nn.functional.pairwise_distance(embed1, embed2, 2)
                    else:
                        outfit_score += metric(embed1 * embed2).detach()

                    num_comparisons += 1.

            if num_comparisons > 0:
                outfit_score /= num_comparisons
            scores.append(outfit_score)

        scores = torch.cat(scores).squeeze().cpu().numpy()
        auc = roc_auc_score(labels, 1 - scores)
        return auc

    def test_fitb(self, embeds, metric):
        correct = 0.
        n_questions = 0.
        for q_index, (questions, answers, is_correct) in enumerate(self.fitb_questions):
            answer_score = np.zeros(len(answers), dtype=np.float32)
            for index, (answer, img1) in enumerate(answers):
                type1 = self.im2type[img1]
                score = 0.0
                for question, img2 in questions:
                    type2 = self.im2type[img2]
                    condition = self.get_typespace(type1, type2)
                    embed1 = embeds[question][condition].unsqueeze(0)
                    embed2 = embeds[answer][condition].unsqueeze(0)

                    if metric is None:
                        score += torch.nn.functional.pairwise_distance(embed1, embed2, 2)
                    else:
                        score += metric(embed1 * embed2).detach()

                answer_score[index] = score.item() if torch.is_tensor(score) else score

            correct += is_correct[np.argmin(answer_score)]
            n_questions += 1

        acc = correct / n_questions
        return acc

    def __getitem__(self, index):
        if self.is_train:
            outfit_id, anchor_im, pos_im = self.pos_pairs[index]
            img1, desc1, has_text1, anchor_type = self.load_train_item(anchor_im)
            img2, desc2, has_text2, item_type = self.load_train_item(pos_im)

            neg_im = self.sample_negative(outfit_id, pos_im, item_type)
            img3, desc3, has_text3, _ = self.load_train_item(neg_im)
            condition = self.get_typespace(anchor_type, item_type)
            return img1, desc1, has_text1, img2, desc2, has_text2, img3, desc3, has_text3, condition

        anchor = self.imnames[index]
        img1 = self.loader(os.path.join(self.impath, '%s.jpg' % anchor))
        if self.transform is not None:
            img1 = self.transform(img1)

        return img1

    def shuffle(self):
        np.random.shuffle(self.pos_pairs)

    def __len__(self):
        if self.is_train:
            return len(self.pos_pairs)
        return len(self.imnames)

Writing polyvore_outfits.py


triple net


In [9]:
%%writefile tripletnet.py
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

def make_fc_1d(f_in, f_out):
    return nn.Sequential(nn.Linear(f_in, f_out),
                         nn.BatchNorm1d(f_out, eps=0.001, momentum=0.01),
                         nn.ReLU(inplace=True))

def selective_margin_loss(pos_samples, neg_samples, margin, has_sample):
    """ pos_samples: Distance between positive pair
        neg_samples: Distance between negative pair
        margin: minimum desired margin between pos and neg samples
        has_sample: Indicates if the sample should be used to calculate the loss
    """
    margin_diff = torch.clamp((pos_samples - neg_samples) + margin, min=0.0, max=1e6)
    # Modern approach to clamping scalar tensor minimums
    num_sample = torch.clamp(torch.sum(has_sample), min=1.0)
    return torch.sum(margin_diff * has_sample) / num_sample

def accuracy(pos_samples, neg_samples):
    """ pos_samples: Distance between positive pair
        neg_samples: Distance between negative pair
    """
    # Native PyTorch tensors handle devices natively now.
    # Calculating the mean of a boolean mask converted to floats yields accuracy.
    pred = (pos_samples - neg_samples)
    acc = (pred > 0).float().mean()
    return acc

class EmbedBranch(nn.Module):
    def __init__(self, feat_dim, embedding_dim):
        super(EmbedBranch, self).__init__()
        self.fc1 = make_fc_1d(feat_dim, embedding_dim)
        self.fc2 = nn.Linear(embedding_dim, embedding_dim)

    def forward(self, x):
        x = self.fc1(x)
        x = self.fc2(x)
        # Modern, native L2 normalization (replaces manual expansion and division)
        return F.normalize(x, p=2, dim=1)

class Tripletnet(nn.Module):
    def __init__(self, args, embeddingnet, text_dim, criterion):
        super(Tripletnet, self).__init__()
        self.embeddingnet = embeddingnet
        self.text_branch = EmbedBranch(text_dim, args.dim_embed)
        self.metric_branch = None

        if args.learned_metric:
            self.metric_branch = nn.Linear(args.dim_embed, 1, bias=False)
            # initilize as having an even weighting across all dimensions
            weight = torch.zeros(1, args.dim_embed) / float(args.dim_embed)
            self.metric_branch.weight = nn.Parameter(weight)

        self.criterion = criterion
        self.margin = args.margin

    def image_forward(self, x, y, z):
        """ x: Anchor data
            y: Distant (negative) data
            z: Close (positive) data
        """
        c = x.conditions
        embedded_x, masknorm_norm_x, embed_norm_x, general_x = self.embeddingnet(x.images, c)
        embedded_y, masknorm_norm_y, embed_norm_y, general_y = self.embeddingnet(y.images, c)
        embedded_z, masknorm_norm_z, embed_norm_z, general_z = self.embeddingnet(z.images, c)

        mask_norm = (masknorm_norm_x + masknorm_norm_y + masknorm_norm_z) / 3
        embed_norm = (embed_norm_x + embed_norm_y + embed_norm_z) / 3
        loss_embed = embed_norm / np.sqrt(len(x))
        loss_mask = mask_norm / len(x)

        if self.metric_branch is None:
            dist_a = F.pairwise_distance(embedded_x, embedded_y, 2)
            dist_b = F.pairwise_distance(embedded_x, embedded_z, 2)
        else:
            dist_a = self.metric_branch(embedded_x * embedded_y)
            dist_b = self.metric_branch(embedded_x * embedded_z)

        # ones_like automatically matches shape and device of dist_a
        target = torch.ones_like(dist_a)

        # type specific triplet loss
        loss_triplet = self.criterion(dist_a, dist_b, target)
        acc = accuracy(dist_a, dist_b)

        # calculate image similarity loss on the general embedding
        disti_p = F.pairwise_distance(general_y, general_z, 2)
        disti_n1 = F.pairwise_distance(general_y, general_x, 2)
        disti_n2 = F.pairwise_distance(general_z, general_x, 2)
        loss_sim_i1 = self.criterion(disti_p, disti_n1, target)
        loss_sim_i2 = self.criterion(disti_p, disti_n2, target)
        loss_sim_i = (loss_sim_i1 + loss_sim_i2) / 2.

        return acc, loss_triplet, loss_sim_i, loss_mask, loss_embed, general_x, general_y, general_z

    def text_forward(self, x, y, z):
        desc_x = self.text_branch(x.text)
        desc_y = self.text_branch(y.text)
        desc_z = self.text_branch(z.text)

        distd_p = F.pairwise_distance(desc_y, desc_z, 2)
        distd_n1 = F.pairwise_distance(desc_x, desc_y, 2)
        distd_n2 = F.pairwise_distance(desc_x, desc_z, 2)

        has_text = x.has_text * y.has_text * z.has_text
        loss_sim_t1 = selective_margin_loss(distd_p, distd_n1, self.margin, has_text)
        loss_sim_t2 = selective_margin_loss(distd_p, distd_n2, self.margin, has_text)
        loss_sim_t = (loss_sim_t1 + loss_sim_t2) / 2.

        return loss_sim_t, desc_x, desc_y, desc_z

    def calc_vse_loss(self, desc_x, general_x, general_y, general_z, has_text):
        distd1_p = F.pairwise_distance(general_x, desc_x, 2)
        distd1_n1 = F.pairwise_distance(general_y, desc_x, 2)
        distd1_n2 = F.pairwise_distance(general_z, desc_x, 2)

        loss_vse_1 = selective_margin_loss(distd1_p, distd1_n1, self.margin, has_text)
        loss_vse_2 = selective_margin_loss(distd1_p, distd1_n2, self.margin, has_text)

        return (loss_vse_1 + loss_vse_2) / 2.

    def forward(self, x, y, z):
        acc, loss_triplet, loss_sim_i, loss_mask, loss_embed, general_x, general_y, general_z = self.image_forward(x, y, z)
        loss_sim_t, desc_x, desc_y, desc_z = self.text_forward(x, y, z)

        loss_vse_x = self.calc_vse_loss(desc_x, general_x, general_y, general_z, x.has_text)
        loss_vse_y = self.calc_vse_loss(desc_y, general_y, general_x, general_z, y.has_text)
        loss_vse_z = self.calc_vse_loss(desc_z, general_z, general_x, general_y, z.has_text)
        loss_vse = (loss_vse_x + loss_vse_y + loss_vse_z) / 3.

        return acc, loss_triplet, loss_mask, loss_embed, loss_vse, loss_sim_t, loss_sim_i

Writing tripletnet.py


type_specific_network.py

In [10]:
%%writefile type_specific_network.py
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np

# Note: The custom ListModule was removed. It is obsolete; PyTorch provides nn.ModuleList natively.

class TypeSpecificNet(nn.Module):
    def __init__(self, args, embeddingnet, n_conditions):
        super(TypeSpecificNet, self).__init__()
        self.learnedmask = args.learned
        prein = args.prein

        if args.rand_typespaces:
            n_conditions = int(np.ceil(n_conditions / float(args.num_rand_embed)))

        self.embeddingnet = embeddingnet
        self.fc_masks = args.use_fc
        self.l2_norm = args.l2_embed

        if self.fc_masks:
            # Modern ModuleList handles iteration and parameter registration securely
            masks = [nn.Linear(args.dim_embed, args.dim_embed) for _ in range(n_conditions)]
            self.masks = nn.ModuleList(masks)
        else:
            self.masks = torch.nn.Embedding(n_conditions, args.dim_embed)

            # Using PyTorch tensors directly for initialization instead of Numpy to avoid bridging quirks
            mask_tensor = torch.zeros((n_conditions, args.dim_embed), dtype=torch.float32)

            if self.learnedmask:
                if prein:
                    mask_tensor.fill_(0.1)
                    mask_len = int(args.dim_embed / n_conditions)
                    for i in range(n_conditions):
                        mask_tensor[i, i*mask_len:(i+1)*mask_len] = 1.0
                    self.masks.weight = torch.nn.Parameter(mask_tensor, requires_grad=True)
                else:
                    nn.init.normal_(self.masks.weight, mean=0.9, std=0.7)
            else:
                mask_len = int(args.dim_embed / n_conditions)
                for i in range(n_conditions):
                    mask_tensor[i, i*mask_len:(i+1)*mask_len] = 1.0
                self.masks.weight = torch.nn.Parameter(mask_tensor, requires_grad=False)

    def forward(self, x, c=None):
        embedded_x = self.embeddingnet(x)

        if c is None:
            # used during testing, wants all type specific embeddings returned for an image
            if self.fc_masks:
                masked_embedding = []
                for mask in self.masks:
                    masked_embedding.append(mask(embedded_x).unsqueeze(1))
                masked_embedding = torch.cat(masked_embedding, 1)
                embedded_x = embedded_x.unsqueeze(1)
            else:
                # Replaced deprecated .data with .detach() to safely separate from autograd graph if needed
                masks = self.masks.weight.detach()
                masks = masks.unsqueeze(0).repeat(embedded_x.size(0), 1, 1)
                embedded_x = embedded_x.unsqueeze(1)
                masked_embedding = embedded_x.expand_as(masks) * masks

            if self.l2_norm:
                masked_embedding = F.normalize(masked_embedding, p=2, dim=2)

            return torch.cat((masked_embedding, embedded_x), 1)

        if self.fc_masks:
            mask_norm = 0.
            masked_embedding = []
            for embed, condition in zip(embedded_x, c):
                 mask = self.masks[condition]
                 masked_embedding.append(mask(embed.unsqueeze(0)))
                 # linalg.vector_norm is the modern standard for torch.norm
                 mask_norm += torch.linalg.vector_norm(mask.weight, ord=1)

            masked_embedding = torch.cat(masked_embedding)
        else:
            self.mask = self.masks(c)
            if self.learnedmask:
                self.mask = F.relu(self.mask)

            masked_embedding = embedded_x * self.mask
            mask_norm = torch.linalg.vector_norm(self.mask, ord=1)

        embed_norm = torch.linalg.vector_norm(embedded_x, ord=2)

        if self.l2_norm:
            masked_embedding = F.normalize(masked_embedding, p=2, dim=1)

        return masked_embedding, mask_norm, embed_norm, embedded_x

Writing type_specific_network.py


main.py

In [13]:
%%writefile main.py
from __future__ import print_function
import random
import argparse
import os
import sys
import shutil
import json

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import transforms
import torch.backends.cudnn as cudnn

# Import your local modules
import Resnet_18
from polyvore_outfits import TripletImageLoader
from tripletnet import Tripletnet
from type_specific_network import TypeSpecificNet

parser = argparse.ArgumentParser(description='Fashion Compatibility Example')

parser.add_argument('--checkpoint_dir', default='/kaggle/working/checkpoints', type=str, help='directory to save checkpoints permanently')
parser.add_argument('--batch-size', type=int, default=512, metavar='N', help='input batch size')
parser.add_argument('--epochs', type=int, default=10, metavar='N', help='number of epochs')
parser.add_argument('--start_epoch', type=int, default=1, metavar='N', help='start epoch')
parser.add_argument('--lr', type=float, default=5e-5, metavar='LR', help='learning rate')
parser.add_argument('--seed', type=int, default=42, metavar='S', help='random seed')
parser.add_argument('--no-cuda', action='store_true', default=False, help='enables CUDA training')
parser.add_argument('--log-interval', type=int, default=250, metavar='N', help='log interval')
parser.add_argument('--resume', default='', type=str, help='path to latest checkpoint')
parser.add_argument('--name', default='Type_Specific_Fashion_Compatibility', type=str, help='experiment name')
parser.add_argument('--polyvore_split', default='nondisjoint', type=str, help='polyvore data split')
parser.add_argument('--datadir', default='data', type=str, help='dataset directory')
parser.add_argument('--test', dest='test', action='store_true', default=False, help='Only run inference')
parser.add_argument('--dim_embed', type=int, default=64, metavar='N', help='embedding dimensions')
parser.add_argument('--use_fc', action='store_true', default=False, help='Use FC layer for embeddings')
parser.add_argument('--learned', dest='learned', action='store_true', default=False, help='learn masks')
parser.add_argument('--prein', dest='prein', action='store_true', default=False, help='initialize masks to be disjoint')
parser.add_argument('--rand_typespaces', action='store_true', default=False, help='randomly assigns comparisons')
parser.add_argument('--num_rand_embed', type=int, default=4, metavar='N', help='number of random embeddings')
parser.add_argument('--l2_embed', dest='l2_embed', action='store_true', default=False, help='L2 normalize')
parser.add_argument('--learned_metric', dest='learned_metric', action='store_true', default=False, help='Learn distance metric')
parser.add_argument('--margin', type=float, default=0.3, metavar='M', help='margin for triplet loss')
parser.add_argument('--embed_loss', type=float, default=5e-4, metavar='M', help='loss for embedding norm')
parser.add_argument('--mask_loss', type=float, default=5e-4, metavar='M', help='loss for mask norm')
parser.add_argument('--vse_loss', type=float, default=5e-3, metavar='M', help='visual-semantic embedding loss')
parser.add_argument('--sim_t_loss', type=float, default=5e-5, metavar='M', help='text-text similarity loss')
parser.add_argument('--sim_i_loss', type=float, default=5e-5, metavar='M', help='image-image similarity loss')

args = None
device = None

# --- NEW WRAPPER TO FIX MULTI-GPU TENSOR SCATTERING ---
class TripletnetWrapper(nn.Module):
    """Wraps Tripletnet so DataParallel can natively scatter raw tensors before wrapping them in TrainData objects."""
    def __init__(self, tnet, use_fc):
        super(TripletnetWrapper, self).__init__()
        self.tnet = tnet
        self.use_fc = use_fc

    def forward(self, img1, desc1, has_text1, condition, img2, desc2, has_text2, img3, desc3, has_text3):
        # Tensors are now safely located on the correct GPU (cuda:0 or cuda:1)
        anchor = TrainData(img1, desc1, has_text1, condition, self.use_fc)
        close = TrainData(img2, desc2, has_text2, None, self.use_fc)
        far = TrainData(img3, desc3, has_text3, None, self.use_fc)
        return self.tnet(anchor, far, close)
# --------------------------------------------------------

def main():
    global args, device
    args = parser.parse_args()

    args.cuda = not args.no_cuda and torch.cuda.is_available()
    device = torch.device("cuda" if args.cuda else "cpu")

    # --- FULL REPRODUCIBILITY LOCKDOWN ---
    seed = args.seed
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if args.cuda:
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
    # --------------------------------------

    normalize = transforms.Normalize(mean=[0.485, 0.456, 0.406],
                                     std=[0.229, 0.224, 0.225])

    fn = os.path.join(args.datadir, 'polyvore_outfits', 'polyvore_item_metadata.json')
    meta_data = json.load(open(fn, 'r'))
    text_feature_dim = 6000
    kwargs = {'num_workers': 2, 'pin_memory': True} if args.cuda else {}

    test_loader = torch.utils.data.DataLoader(
        TripletImageLoader(args, 'test', meta_data,
                           transform=transforms.Compose([
                               transforms.Resize(112),
                               transforms.CenterCrop(112),
                               transforms.ToTensor(),
                               normalize,
                           ])),
        batch_size=args.batch_size, shuffle=False, **kwargs)

    model = Resnet_18.resnet18(pretrained=True, embedding_size=args.dim_embed)
    csn_model = TypeSpecificNet(args, model, len(test_loader.dataset.typespaces))

    criterion = torch.nn.MarginRankingLoss(margin=args.margin)
    
    # Wrap the original model
    tnet = Tripletnet(args, csn_model, text_feature_dim, criterion)
    tnet = TripletnetWrapper(tnet, args.use_fc)
    
    # Distribute across GPUs
    if args.cuda and torch.cuda.device_count() > 1:
        print(f"\n🔥 Multi-GPU Detected! Utilizing {torch.cuda.device_count()} GPUs for parallel training.\n")
        tnet = nn.DataParallel(tnet)
    
    tnet = tnet.to(device)

    train_loader = torch.utils.data.DataLoader(
        TripletImageLoader(args, 'train', meta_data,
                           text_dim=text_feature_dim,
                           transform=transforms.Compose([
                               transforms.Resize(112),
                               transforms.CenterCrop(112),
                               transforms.RandomHorizontalFlip(),
                               transforms.ToTensor(),
                               normalize,
                           ])),
        batch_size=args.batch_size, shuffle=True, **kwargs)

    val_loader = torch.utils.data.DataLoader(
        TripletImageLoader(args, 'valid', meta_data,
                           transform=transforms.Compose([
                               transforms.Resize(112),
                               transforms.CenterCrop(112),
                               transforms.ToTensor(),
                               normalize,
                           ])),
        batch_size=args.batch_size, shuffle=False, **kwargs)

    parameters = filter(lambda p: p.requires_grad, tnet.parameters())
    optimizer = optim.Adam(parameters, lr=args.lr)

    best_acc = 0
    resume_path = args.resume if args.resume else os.path.join(args.checkpoint_dir, args.name, 'checkpoint.pth.tar')

    if os.path.isfile(resume_path):
        print("=> Loading checkpoint '{}'".format(resume_path))
        checkpoint = torch.load(resume_path, map_location=device, weights_only=False)
        args.start_epoch = checkpoint['epoch']
        best_acc = checkpoint['best_prec1']
        
        # Unpack the nested wrappers to load the raw model correctly
        model_to_load = tnet.module.tnet if isinstance(tnet, nn.DataParallel) else tnet.tnet
        model_to_load.load_state_dict(checkpoint['state_dict'])

        if 'optimizer' in checkpoint:
            optimizer.load_state_dict(checkpoint['optimizer'])
            print("=> Loaded optimizer state.")

        if 'torch_rng_state' in checkpoint:
            torch.set_rng_state(checkpoint['torch_rng_state'])
        if args.cuda and 'cuda_rng_state' in checkpoint and checkpoint['cuda_rng_state'] is not None:
            torch.cuda.set_rng_state_all(checkpoint['cuda_rng_state'])

        print("=> Loaded checkpoint '{}' (resuming from epoch {})".format(resume_path, args.start_epoch))
    else:
        print("=> No existing checkpoint found. Starting training from scratch.")

    if args.test:
        test_acc = test(test_loader, tnet)
        sys.exit()

    if args.start_epoch > 1:
        adjust_learning_rate(optimizer, args.start_epoch - 1)

    n_parameters = sum([p.numel() for p in tnet.parameters()])
    print('  + Number of params: {}'.format(n_parameters))

    for epoch in range(args.start_epoch, args.epochs + 1):
        adjust_learning_rate(optimizer, epoch)
        train(train_loader, tnet, criterion, optimizer, epoch)
        acc = test(val_loader, tnet)

        is_best = acc > best_acc
        best_acc = max(acc, best_acc)

        model_to_save = tnet.module.tnet if isinstance(tnet, nn.DataParallel) else tnet.tnet
        
        save_checkpoint({
            'epoch': epoch + 1,
            'state_dict': model_to_save.state_dict(),
            'optimizer': optimizer.state_dict(),
            'best_prec1': best_acc,
            'torch_rng_state': torch.get_rng_state(),
            'cuda_rng_state': torch.cuda.get_rng_state_all() if args.cuda else None,
        }, is_best)

    best_model_path = os.path.join(args.checkpoint_dir, args.name, 'model_best.pth.tar')
    if os.path.exists(best_model_path):
        checkpoint = torch.load(best_model_path, map_location=device, weights_only=False)
        model_to_load = tnet.module.tnet if isinstance(tnet, nn.DataParallel) else tnet.tnet
        model_to_load.load_state_dict(checkpoint['state_dict'])
        test_acc = test(test_loader, tnet)

def train(train_loader, tnet, criterion, optimizer, epoch):
    losses = AverageMeter()
    accs = AverageMeter()
    emb_norms = AverageMeter()
    mask_norms = AverageMeter()

    tnet.train()
    for batch_idx, (img1, desc1, has_text1, img2, desc2, has_text2, img3, desc3, has_text3, condition) in enumerate(train_loader):
        
        # Manually move tensors to primary device before DataParallel scatters them
        if args.cuda:
            img1, desc1, has_text1 = img1.to(device), desc1.to(device), has_text1.to(device)
            img2, desc2, has_text2 = img2.to(device), desc2.to(device), has_text2.to(device)
            img3, desc3, has_text3 = img3.to(device), desc3.to(device), has_text3.to(device)
            condition = condition.to(device)

        # Pass raw tensors directly into the wrapped model
        acc, loss_triplet, loss_mask, loss_embed, loss_vse, loss_sim_t, loss_sim_i = tnet(
            img1, desc1, has_text1, condition, 
            img2, desc2, has_text2, 
            img3, desc3, has_text3
        )

        loss_triplet = loss_triplet.mean()
        loss_mask = loss_mask.mean()
        loss_embed = loss_embed.mean()
        loss_vse = loss_vse.mean()
        loss_sim_t = loss_sim_t.mean()
        loss_sim_i = loss_sim_i.mean()
        acc = acc.mean()

        loss_sim = args.sim_t_loss * loss_sim_t + args.sim_i_loss * loss_sim_i
        loss_vse_w = args.vse_loss * loss_vse
        loss_reg = args.embed_loss * loss_embed + args.mask_loss * loss_mask

        loss = loss_triplet + loss_reg
        if args.vse_loss > 0:
            loss += loss_vse_w
        if args.sim_t_loss > 0 or args.sim_i_loss > 0:
            loss += loss_sim

        num_items = img1.size(0)

        losses.update(loss_triplet.item(), num_items)
        accs.update(acc.item(), num_items)
        emb_norms.update(loss_embed.item())
        mask_norms.update(loss_mask.item())

        optimizer.zero_grad()

        if not torch.isnan(loss):
            loss.backward()
            optimizer.step()

        if batch_idx % args.log_interval == 0:
            print('Train Epoch: {} [{}/{}]\t'
                  'Loss: {:.4f} ({:.4f}) \t'
                  'Acc: {:.2f}% ({:.2f}%) \t'
                  'Emb_Norm: {:.2f} ({:.2f})'.format(
                epoch, batch_idx * num_items, len(train_loader.dataset),
                losses.val, losses.avg,
                100. * accs.val, 100. * accs.avg, emb_norms.val, emb_norms.avg))

@torch.no_grad()
def test(test_loader, tnet):
    tnet.eval()
    embeddings = []
    
    eval_model = tnet.module.tnet if isinstance(tnet, nn.DataParallel) else tnet.tnet

    for batch_idx, images in enumerate(test_loader):
        images = images.to(device)
        embeddings.append(eval_model.embeddingnet(images).detach().cpu())

    embeddings = torch.cat(embeddings)
    metric = eval_model.metric_branch

    auc = test_loader.dataset.test_compatibility(embeddings, metric)
    acc = test_loader.dataset.test_fitb(embeddings, metric)
    total = auc + acc
    print('\n{} set: Compat AUC: {:.2f} FITB: {:.1f}\n'.format(
        test_loader.dataset.split,
        round(auc, 2), round(acc * 100, 1)))

    return total

def save_checkpoint(state, is_best, filename='checkpoint.pth.tar'):
    directory = os.path.join(args.checkpoint_dir, args.name)
    if not os.path.exists(directory):
        os.makedirs(directory)

    filepath = os.path.join(directory, filename)
    torch.save(state, filepath)
    if is_best:
        best_filepath = os.path.join(directory, 'model_best.pth.tar')
        shutil.copyfile(filepath, best_filepath)
        print(f"--> Saved new best model to Kaggle Working Directory: {best_filepath}")

class TrainData():
    # MODIFIED: Removed the hardcoded `.to(device)` so tensors can live naturally on GPU 0 or 1
    def __init__(self, images, text, has_text, conditions=None, use_fc=False):
        self.has_text = has_text.float()
        self.images = images
        self.text = text

        if conditions is not None and not use_fc:
            self.conditions = conditions
        else:
            self.conditions = None

    def __len__(self):
        return self.images.size(0)

class AverageMeter(object):
    def __init__(self):
        self.reset()

    def reset(self):
        self.val = 0
        self.avg = 0
        self.sum = 0
        self.count = 0

    def update(self, val, n=1):
        self.val = val
        self.sum += val * n
        self.count += n
        self.avg = self.sum / self.count

def adjust_learning_rate(optimizer, epoch):
    lr = args.lr * ((1 - 0.015) ** epoch)
    for param_group in optimizer.param_groups:
        param_group['lr'] = lr

if __name__ == '__main__':
    main()

Writing main.py


In [14]:
!python main.py \
  --name polyvore_run_full \
  --learned \
  --l2_embed \
  --batch-size 720 \
  --checkpoint_dir /kaggle/working/checkpoints \
  --sim_t_loss 0 \
  --vse_loss 0

Downloading: "https://download.pytorch.org/models/resnet18-5c106cde.pth" to /root/.cache/torch/hub/checkpoints/resnet18-5c106cde.pth
100%|███████████████████████████████████████| 44.7M/44.7M [00:00<00:00, 145MB/s]

🔥 Multi-GPU Detected! Utilizing 2 GPUs for parallel training.

=> No existing checkpoint found. Starting training from scratch.
  + Number of params: 3191808
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]
Train Epoch: 1 [0/686851]	Loss: 0.2856 (0.2856) 	Acc: 53.61% (53.61%) 	Emb_Norm: 0.77 (0.77)
Train Epoch: 1 [180000/686851]	Loss: 0.2249 (0.2297) 	Acc: 64.58% (65.63%) 	Emb_Norm: 0.69 (0.69)
Train Epoch: 1 [360000/686851]	Loss: 0.2140 (0.2218) 	Acc: 66.53% (66.94%) 	Emb_Norm: 0.71 (0.70)
Train Epoch: 1 [540000/686851]	Loss: 0.2142 (0.2173) 	Acc: 67.64% (67.64